# BigWig Region Visualization — Evolutionary Peaks per Branch
Plots every peak from each evolutionary branch (same regions as `12_heatmap.ipynb`) using
filtered species bigwigs (`bigwigs_filter/enterocytes/`). Each peak is saved as an individual
PNG under a subfolder named after its branch.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys, re

sys.path.insert(0, os.path.abspath(".."))
from src.utils import resolve_path
from src.visualization import plot_genome_regions

# ── Paths (Euler-style; auto-resolved on Treutlein server) ────────────────────
BASE       = resolve_path("/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks")
CONS_DIR   = f"{BASE}/cross_species_consensus_v3"
DESEQ_DIR  = f"{CONS_DIR}/13_deseq2_R_pseudobulk"
ANNO_PATH  = f"{CONS_DIR}/07_master_annotation/master_annotation_corrected.tsv"
FRAG_DIR   = f"{CONS_DIR}/12_fragment_matrices"
ROBUST_DIR = f"{DESEQ_DIR}/differential_results/ultra_robust_branches"
BRANCH_DIR = f"{DESEQ_DIR}/differential_results/evolutionary_branches"

SPECIES       = ["Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset"]
CELL_TYPE     = "Enterocytes"
MAX_PER_BLOCK = 50

# ── Load annotation ───────────────────────────────────────────────────────────
print("Loading annotation...")
anno = pd.read_csv(ANNO_PATH, sep="\t", low_memory=False).set_index("peak_id")
orth_cols = {sp: f"{sp}_orth" for sp in SPECIES}
det_cols  = {sp: f"{sp}_det"  for sp in SPECIES}
print(f"  {anno.shape[0]:,} peaks x {anno.shape[1]} columns")

Loading annotation...
  1,142,003 peaks x 59 columns


In [9]:
# ── Load species-level pseudobulk accessibility (needed for waterfall filters) ─
acc_species = {}

for sp in SPECIES:
    counts_path = resolve_path(f"{FRAG_DIR}/{sp}/pseudobulk_counts.parquet")
    groups_path = resolve_path(f"{FRAG_DIR}/{sp}/pseudobulk_groups.parquet")
    if not os.path.exists(counts_path):
        counts_path = counts_path.replace(".parquet", ".tsv")
        groups_path = groups_path.replace(".parquet", ".tsv")

    if groups_path.endswith(".parquet"):
        groups = pd.read_parquet(groups_path)
    else:
        groups = pd.read_csv(groups_path, sep="\t")

    ct_groups = groups[groups["cell_type"] == CELL_TYPE].copy()
    if len(ct_groups) == 0:
        print(f"  {sp}: no {CELL_TYPE} samples, skipping")
        continue

    labels = ct_groups["label"].tolist()
    print(f"  {sp}: {len(labels)} {CELL_TYPE} pseudobulks")

    if counts_path.endswith(".parquet"):
        df = pd.read_parquet(counts_path)
        if "region_id" in df.columns:
            df = df.set_index("region_id")
    else:
        df = pd.read_csv(counts_path, sep="\t", index_col=0)

    valid_labels = [l for l in labels if l in df.columns]
    if len(valid_labels) == 0:
        continue
    sub = df[valid_labels]

    mean_counts = sub.mean(axis=1)
    total = mean_counts.sum()
    cpm = (mean_counts / total * 1e6) if total > 0 else mean_counts
    acc_species[sp] = np.log2(cpm + 1)

filter_acc_df = pd.DataFrame(acc_species)
print(f"\nSpecies matrix: {filter_acc_df.shape[0]:,} peaks x {filter_acc_df.shape[1]} columns")

  Human: 5 Enterocytes pseudobulks
  Bonobo: 2 Enterocytes pseudobulks
  Chimpanzee: 3 Enterocytes pseudobulks
  Gorilla: 3 Enterocytes pseudobulks
  Macaque: 1 Enterocytes pseudobulks
  Marmoset: 2 Enterocytes pseudobulks

Species matrix: 1,142,003 peaks x 6 columns


In [10]:
# ── Helper functions (same as 12_heatmap.ipynb) ───────────────────────────────

def is_true(series):
    """Coerce mixed bool/str/int column to boolean mask."""
    if series.dtype == bool:
        return series.fillna(False)
    if series.dtype == object:
        return series.astype(str).str.strip().str.upper().isin(["TRUE", "1"])
    return series.fillna(0).astype(bool)

def load_robust(contrast):
    """Load ultra-robust peak IDs for CELL_TYPE."""
    path = f"{ROBUST_DIR}/{CELL_TYPE}/{contrast}_UltraRobust.csv"
    if not os.path.exists(path):
        print(f"  [MISSING robust] {contrast}")
        return set()
    return set(pd.read_csv(path).iloc[:, 0].astype(str))

def load_branch_df(contrast):
    """Load full DESeq2 branch result for CELL_TYPE."""
    path = f"{BRANCH_DIR}/{CELL_TYPE}/{contrast}.csv"
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    id_col = "peak_id" if "peak_id" in df.columns else df.columns[0]
    return df.set_index(id_col)

def load_branch_sig(contrast, padj=0.05, lfc=1.0):
    """Load significant UP peak IDs from branch result."""
    df = load_branch_df(contrast)
    if df is None:
        print(f"  [MISSING branch] {contrast}")
        return set()
    df = df.dropna(subset=["padj", "log2FoldChange"])
    sig = df[(df["padj"] < padj) & (df["log2FoldChange"] > lfc)]
    return set(sig.index.astype(str))

def ortho_filter(peaks, required_species):
    """Keep only peaks with orthologous sequence in ALL required species."""
    mask = anno[[orth_cols[s] for s in required_species]].apply(lambda c: is_true(c)).all(axis=1)
    return set(peaks) & set(anno.index[mask])

def rank_peaks(peak_set, contrast, n=MAX_PER_BLOCK):
    """Rank peaks by |LFC| x -log10(padj)."""
    df = load_branch_df(contrast)
    if df is None:
        out = list(peak_set)
        return out if n is None else out[:n]
    df = df.loc[df.index.intersection(peak_set)].dropna(subset=["padj", "log2FoldChange"])
    df["score"] = df["log2FoldChange"].abs() * (-np.log10(df["padj"].clip(lower=1e-300)))
    ranked = list(df.sort_values("score", ascending=False).index)
    return ranked if n is None else ranked[:n]

def rank_by_basemean(peak_set, n=MAX_PER_BLOCK):
    """Rank peaks by baseMean from available branch contrasts."""
    for contrast in ["Div_Human_vs_Apes", "Node1_Human_vs_Pan"]:
        df = load_branch_df(contrast)
        if df is not None:
            overlap = df.index.intersection(peak_set)
            if len(overlap) > 0:
                ranked = list(df.loc[overlap].sort_values("baseMean", ascending=False).index)
                return ranked if n is None else ranked[:n]
    if "Human" in filter_acc_df.columns:
        overlap = filter_acc_df.index.intersection(peak_set)
        ranked = list(filter_acc_df.loc[overlap, "Human"].sort_values(ascending=False).index)
        return ranked if n is None else ranked[:n]
    out = list(peak_set)
    return out if n is None else out[:n]

def acc_waterfall(peak_set, up_species, down_species):
    """Strict waterfall: down-species (with orthology) must stay below up-species."""
    peaks = sorted(set(peak_set) & set(filter_acc_df.index))
    if not peaks:
        return set()
    sub = filter_acc_df.loc[peaks]
    up_min = sub[up_species].min(axis=1)
    keep = pd.Series(True, index=peaks)
    for sp in down_species:
        has_orth = is_true(anno.reindex(peaks)[orth_cols[sp]])
        sp_val = sub[sp].fillna(-999)
        keep &= ~(has_orth & (sp_val >= up_min))
    return set(pd.Index(peaks)[keep])

def posthoc_filter(peak_list, up_species, down_species):
    """Extra strict clean-up: remove peaks if any down-species reaches >80% of up mean."""
    if not peak_list:
        return peak_list
    peaks = [p for p in peak_list if p in filter_acc_df.index]
    sub = filter_acc_df.loc[peaks]
    up_mean = sub[up_species].mean(axis=1)
    keep = pd.Series(True, index=peaks)
    for sp in down_species:
        has_orth = is_true(anno.reindex(peaks)[orth_cols[sp]])
        sp_val = sub[sp].fillna(0)
        keep &= ~(has_orth & (sp_val > up_mean * 0.8))
    return [p for p in peaks if keep[p]]

print("Helpers loaded.")

Helpers loaded.


In [ ]:
# ── Build regions dictionary (same logic as 12_heatmap.ipynb) ─────────────────

other_sp = [s for s in SPECIES if s != "Human"]
regions = {}
regions_all = {}
total_candidates = {}

# Pre-load ultra-robust sets
robust = {}
for c in [
    "Node4_OldWorld_vs_Marmoset",
    "Node3_GreatApes_vs_Macaque",
    "ILS_HumanGorilla_vs_Pan",
    "ILS_HumanChimp_vs_GorillaBonobo",
    "ILS_HumanBonobo_vs_ChimpGorilla",
    "Div_Human_vs_Apes",
    "Node1_Human_vs_Pan",
]:
    robust[c] = load_robust(c)

# Pre-load significant peaks for hierarchical subtraction
sig_node4 = load_branch_sig("Node4_OldWorld_vs_Marmoset", padj=0.1, lfc=0.5)
sig_node3 = load_branch_sig("Node3_GreatApes_vs_Macaque", padj=0.1, lfc=0.5)
deeper_than_ils = (robust["Node3_GreatApes_vs_Macaque"] | robust["Node4_OldWorld_vs_Marmoset"] | sig_node3 | sig_node4)
deeper_than_n3  = (robust["Node4_OldWorld_vs_Marmoset"] | sig_node4)

# 1. Human-Specific DNA
human_spec_mask = (
    is_true(anno[orth_cols["Human"]])
    & anno[[orth_cols[s] for s in other_sp]].apply(lambda c: ~is_true(c)).all(axis=1)
    & is_true(anno[det_cols["Human"]])
)
human_spec_all = set(anno.index[human_spec_mask])
ranked_all = rank_by_basemean(human_spec_all, n=None)
regions_all["Human-specific DNA"] = ranked_all
regions["Human-specific DNA"] = ranked_all[:MAX_PER_BLOCK]
total_candidates["Human-specific DNA"] = len(ranked_all)
print(f"  Human-specific DNA:                {len(ranked_all):>5} -> {len(regions['Human-specific DNA'])} shown")

# 2-4. ILS contrasts
ils_defs = [
    ("ILS: Human + Gorilla UP",      "ILS_HumanGorilla_vs_Pan",          ["Human", "Gorilla"],     ["Chimpanzee", "Bonobo"]),
    ("ILS: Human + Chimpanzee UP",   "ILS_HumanChimp_vs_GorillaBonobo",  ["Human", "Chimpanzee"],  ["Gorilla", "Bonobo"]),
    ("ILS: Human + Bonobo UP",       "ILS_HumanBonobo_vs_ChimpGorilla",  ["Human", "Bonobo"],      ["Chimpanzee", "Gorilla"]),
]
for label, contrast, up_sp, contrast_down_sp in ils_defs:
    filtered = ortho_filter(robust[contrast], up_sp + contrast_down_sp)
    filtered -= deeper_than_ils
    all_down = contrast_down_sp + ["Macaque", "Marmoset"]
    filtered = acc_waterfall(filtered, up_sp, all_down)
    ranked_all = rank_peaks(filtered, contrast, n=None)
    ranked_all = posthoc_filter(ranked_all, up_sp, all_down)
    regions_all[label] = ranked_all
    regions[label] = ranked_all[:MAX_PER_BLOCK]
    total_candidates[label] = len(ranked_all)
    print(f"  {label:<35} {len(ranked_all):>5} -> {len(regions[label])} shown")

# 5. Great Apes UP vs Macaque
n3_filtered = ortho_filter(
    robust["Node3_GreatApes_vs_Macaque"],
    ["Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque"],
)
n3_filtered -= deeper_than_n3
n3_filtered = acc_waterfall(n3_filtered, ["Human", "Chimpanzee", "Bonobo", "Gorilla"], ["Macaque", "Marmoset"])
ranked_all = rank_peaks(n3_filtered, "Node3_GreatApes_vs_Macaque", n=None)
ranked_all = posthoc_filter(ranked_all, ["Human", "Chimpanzee", "Bonobo", "Gorilla"], ["Macaque", "Marmoset"])
regions_all["Great Apes UP vs Macaque"] = ranked_all
regions["Great Apes UP vs Macaque"] = ranked_all[:MAX_PER_BLOCK]
total_candidates["Great Apes UP vs Macaque"] = len(ranked_all)
print(f"  Great Apes UP vs Macaque:          {len(ranked_all):>5} -> {len(regions['Great Apes UP vs Macaque'])} shown")

# 6. Old World UP vs Marmoset
n4_filtered = ortho_filter(robust["Node4_OldWorld_vs_Marmoset"], SPECIES)
n4_filtered = acc_waterfall(n4_filtered, ["Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque"], ["Marmoset"])
ranked_all = rank_peaks(n4_filtered, "Node4_OldWorld_vs_Marmoset", n=None)
ranked_all = posthoc_filter(ranked_all, ["Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque"], ["Marmoset"])
regions_all["Old World UP vs Marmoset"] = ranked_all
regions["Old World UP vs Marmoset"] = ranked_all[:MAX_PER_BLOCK]
total_candidates["Old World UP vs Marmoset"] = len(ranked_all)
print(f"  Old World UP vs Marmoset:          {len(ranked_all):>5} -> {len(regions['Old World UP vs Marmoset'])} shown")

print(f"\nTotal regions: {sum(len(v) for v in regions.values())} peaks across {len(regions)} branches")

In [12]:
# ── Cell-type colours ─────────────────────────────────────────────────────────
CELL_TYPE_COLORS = {
    'Enterocytes':                          '#D95B27',
    'Colonocytes':                          '#DA5C2D',
    'TA cells':                             '#F5865F',
    'BEST4 cells':                          '#913C1F',
    'Fibroblasts SYNM+':                    '#E59825',
    'Fibroblasts RSPO2/3+':                 '#FCB137',
    'Myofibroblasts':                       '#AA732A',
    'Fibroblasts PCDH9+':                   '#A8762B',
    'Fibroblasts RALYL+':                   '#8C411E',
    'Fibroblasts RSPO3+':                   '#FECC81',
    'Crypt Fibroblasts WNT2B+':             '#AB7B2C',
    'Enteric glia':                         '#F05653',
    'Fibroblasts ADAMTS16+':               '#533B16',
    'Villus Fibroblasts WNT5B+':            '#FDC172',
    'Fibroblasts VCAM1+':                   '#F7AF1A',
    'Fibroblasts KCNN3+':                   '#F4C068',
    'ICCs':                                 '#DA9C28',
    'Goblet cells':                         '#A14223',
    'Paneth cells':                         '#F7956F',
    'Stem cells':                           '#F6967B',
    'Tuft cells':                           '#F8A488',
    'Enteric neurons':                      '#93292A',
    'Pericytes':                            '#FECE92',
    'Mast cells':                           '#872890',
    'Mesothelial cells':                    '#FDD7A4',
    'ECs':                                  '#327338',
    'NK/ILCs':                              '#822989',
    'Eosinophils':                          '#B975B1',
    'Neutrophils':                          '#852990',
    'T cells':                              '#CD7AB2',
    'Lymphatic ECs':                        '#64BB67',
    'RBP2 high cells':                      '#8E191D',
    'MARCO+ Lymphatic ECs':                 '#238842',
    'Adipocytes':                           '#895422',
    'Naive B Cells':                        '#A9459A',
    'MUC6+ cells':                          '#D1643D',
    'EECs':                                 '#B25A27',
}

# ── Broad-class colours ───────────────────────────────────────────────────────
BROAD_CLASS_COLORS = {
    'Epithelial':   '#D24627',
    'Immune':       '#A54A9C',
    'Neuronal':     '#D42027',
    'Endothelial':  '#377639',
    'Mesenchymal':  '#D69E28',
}

# ── Species colours ───────────────────────────────────────────────────────────
SPECIES_COLORS = {
    'Human':       '#43A047',
    'Chimpanzee':  '#FCB216',
    'Bonobo':      '#1A889D',
    'Gorilla':     '#892782',
    'Macaque':     '#EF476F',
    'Marmoset':    '#F36F21',
}

# ── Region colours ────────────────────────────────────────────────────────────
REGION_COLORS = {
    'Duodenum': '#009481',
    'Colon':    '#C33956',
}

# ── BigWig paths (server-aware) ───────────────────────────────────────────────
# Files: {lowercase_species}_enterocytes.hg38.cov.bw
BW_BASE = resolve_path(
    "/cluster/home/jjanssens/jjans/analysis/adult_intestine/dl_models"
    "/enterocytes_v0/bigwigs_filter/enterocytes"
)

bw_species_order = ["Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset"]
bigwig_files = [f"{BW_BASE}/{sp.lower()}_enterocytes.hg38.cov.bw" for sp in bw_species_order]
bw_colors    = [SPECIES_COLORS[sp] for sp in bw_species_order]

print(f"BigWig base: {BW_BASE}")
for f, sp in zip(bigwig_files, bw_species_order):
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f"  [{status}] {sp}: {os.path.basename(f)}")

BigWig base: /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/dl_models/enterocytes_v0/bigwigs_filter/enterocytes
  [OK] Human: human_enterocytes.hg38.cov.bw
  [OK] Bonobo: bonobo_enterocytes.hg38.cov.bw
  [OK] Chimpanzee: chimpanzee_enterocytes.hg38.cov.bw
  [OK] Gorilla: gorilla_enterocytes.hg38.cov.bw
  [OK] Macaque: macaque_enterocytes.hg38.cov.bw
  [OK] Marmoset: marmoset_enterocytes.hg38.cov.bw


In [14]:
# ── Plot each region separately, organised by branch ─────────────────────────
# Output structure:
#   DESEQ_DIR/bigwig_regions/{branch_folder}/{peak_id}.png
#
# ymax is auto-detected per region: 99th percentile across all bigwig tracks,
# rounded up to the nearest 5 (minimum 5).

import pyBigWig

def compute_ymax(bigwig_files, chrom, start, end, percentile=99, min_ymax=5, round_to=5):
    """Return a rounded ymax derived from the signal in all bigwig tracks for this region."""
    all_vals = []
    for bw_path in bigwig_files:
        if not os.path.exists(bw_path):
            continue
        try:
            bw = pyBigWig.open(bw_path)
            chrom_bw = chrom if chrom in bw.chroms() else (
                "chr" + chrom if "chr" + chrom in bw.chroms() else chrom
            )
            vals = bw.values(chrom_bw, start, end)
            bw.close()
            if vals:
                all_vals.extend(np.nan_to_num(np.array(vals)))
        except Exception:
            pass
    if not all_vals:
        return float(min_ymax)
    peak_val = np.percentile(all_vals, percentile)
    return float(max(min_ymax, np.ceil(peak_val / round_to) * round_to))


OUTPUT_BASE = os.path.join(DESEQ_DIR, "bigwig_regions")
os.makedirs(OUTPUT_BASE, exist_ok=True)

def _safe_folder(name):
    """Sanitize branch name for use as a directory name."""
    return re.sub(r"[^A-Za-z0-9_+.-]", "_", name).strip("_")

skipped_total = 0
plotted_total = 0

for branch, peak_ids in regions.items():
    folder = os.path.join(OUTPUT_BASE, _safe_folder(branch))
    os.makedirs(folder, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"Branch: {branch}  ({len(peak_ids)} peaks)")
    print(f"Output: {folder}")

    branch_plotted = 0
    branch_skipped = 0

    for peak_id in peak_ids:
        if peak_id not in anno.index:
            branch_skipped += 1
            continue

        row   = anno.loc[peak_id]
        chrom = row.get("Human_chr")
        start = row.get("Human_start")
        end   = row.get("Human_end")

        if pd.isna(chrom) or pd.isna(start) or pd.isna(end):
            print(f"  [SKIP] {peak_id}: no Human coordinates")
            branch_skipped += 1
            continue

        start, end = int(start), int(end)
        coord_str  = f"{chrom}:{start}-{end}"
        out_path   = os.path.join(folder, f"{peak_id}.png")

        ymax = compute_ymax(bigwig_files, chrom, start, end)

        plot_genome_regions(
            bigwigs=bigwig_files,
            selected_regions={peak_id: coord_str},
            colors=bw_colors,
            track_names=bw_species_order,
            ymax=ymax,
            smooth_window=20,
            figsize=(5, len(bw_species_order) * 1.5),
            saveas=out_path,
        )
        plt.close("all")
        branch_plotted += 1

    print(f"  Plotted: {branch_plotted}  |  Skipped (no coords): {branch_skipped}")
    plotted_total += branch_plotted
    skipped_total += branch_skipped

print(f"\n{'='*60}")
print(f"Done. Total plotted: {plotted_total}  |  Total skipped: {skipped_total}")
print(f"Output root: {OUTPUT_BASE}")


Branch: Human-specific DNA  (50 peaks)
Output: /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/bigwig_regions/Human-specific_DNA
  Plotted: 50  |  Skipped (no coords): 0

Branch: ILS: Human + Gorilla UP  (50 peaks)
Output: /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/bigwig_regions/ILS__Human_+_Gorilla_UP
  Plotted: 50  |  Skipped (no coords): 0

Branch: ILS: Human + Chimp UP  (50 peaks)
Output: /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/bigwig_regions/ILS__Human_+_Chimp_UP
  Plotted: 50  |  Skipped (no coords): 0

Branch: ILS: Human + Bonobo UP  (50 peaks)
Output: /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/bigwig_regions/ILS__Human_+_Bonobo_UP
  Plotted: 50  |  Skipped (no coords): 0

Branch: Great A